# LangGraph Agentic English → Hindi Translation System V2

### New additions over V1:
- **Direction 1** — NLP Enrichment: Stanza NER, TIMEX temporal tagging, WordNet sense disambiguation, DBpedia entity linking
- **Direction 2** — Knowledge Graph: NetworkX KG with entities, hypernyms, hyponyms, temporal nodes, UD parses
- **Direction 3** — Multi-Agent MCP: Specialist agents (NER, KG, Translator, Evaluator, Feedback, Memory) with orchestrator
- **Direction 4** — Low Resource: Back-translation pipeline for synthetic dataset generation (Bhojpuri, Maithili, etc.)

**Runtime:** GPU (T4) · **Setup time:** ~8 min

## Step 1 — Install All Dependencies

In [ ]:
# Core LangGraph + LangChain
!pip install -q langgraph>=0.2.0 langchain>=0.3.0 langchain-core>=0.3.0 langchain-groq>=0.1.0

# NLP enrichment tools
!pip install -q stanza trankit nltk spacy
!pip install -q wikidata sparqlwrapper  # DBpedia / Wikidata linking

# Embeddings + Vector DB
!pip install -q sentence-transformers chromadb sqlitedict

# Knowledge Graph
!pip install -q networkx pyvis  # lightweight graph (Neo4j alternative for Colab)

# Evaluation metrics
!pip install -q sacrebleu bert-score

# Fine-tuning stack
!pip install -q transformers datasets peft trl bitsandbytes accelerate

# Utilities
!pip install -q loguru python-dotenv indic-transliteration

# Download Stanza English + Hindi models
import stanza
stanza.download('en', processors='tokenize,pos,lemma,depparse,ner')
stanza.download('hi', processors='tokenize,pos,lemma,depparse,ner')

# Download NLTK WordNet
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

print('✅ All dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.4/773.4 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading these customized packages for language: en (English)...
| Processor       | Package                   |
-----------------------------------------------
| tokenize        | combined                  |
| mwt             | combined                  |
| pos             | combined_charlm           |
| lemma           | combined_nocharlm         |
| depparse        | combined_charlm           |
| ner             | ontonotes-ww-multi_charlm |
| forward_charlm  | 1billion                  |
| backward_charlm | 1billion                  |
| pretrain        | conll17                   |



INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/tokenize/combined.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/mwt/combined.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/pos/combined_charlm.pt


...

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/hi/backward_charlm/oscar.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/hi/pretrain/conll17.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/hi/forward_charlm/oscar.pt
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ All dependencies installed


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


## Step 2 — Configuration & API Key

In [ ]:
import os
from getpass import getpass

# Get free Groq key from: https://console.groq.com/keys
os.environ['GROQ_API_KEY']       = getpass('Enter your Groq API key: ')
os.environ['TRANSLATION_MODEL']  = 'llama-3.3-70b-versatile'
os.environ['PASS_THRESHOLD']     = '0.60'
os.environ['W_BLEU']             = '0.20'
os.environ['W_CHRF']             = '0.25'
os.environ['W_BERT']             = '0.30'
os.environ['W_COMET']            = '0.25'

# Low-resource language config
PIVOT_LANG        = 'hi'   # Hindi as pivot for back-translation
LOW_RESOURCE_LANG = 'bho'  # Bhojpuri (change to 'mai'=Maithili, 'awa'=Awadhi)

print('✅ Configuration set')
print(f'   Model    : {os.environ["TRANSLATION_MODEL"]}')
print(f'   Threshold: {os.environ["PASS_THRESHOLD"]}')

Enter your Groq API key: ··········
✅ Configuration set
   Model    : llama-3.3-70b-versatile
   Threshold: 0.60


## Step 3 — Expanded State Schema

In [ ]:
from __future__ import annotations
from typing import Annotated, Any, Optional
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class EvalScores(TypedDict, total=False):
    bleu: float
    chrf: float
    bert_score_f1: float
    comet: float
    aggregate: float


class NamedEntity(TypedDict, total=False):
    text: str         # surface form e.g. 'Supreme Court'
    type: str         # PERSON, ORG, LOC, DATE, EVENT
    start: int
    end: int
    hindi: str        # resolved Hindi translation from KG
    dbpedia_uri: str  # linked DBpedia resource
    wikidata_id: str  # linked Wikidata QID


class TemporalExpression(TypedDict, total=False):
    text: str         # surface form e.g. 'next Tuesday'
    type: str         # DATE, TIME, DURATION, SET
    value: str        # normalised ISO e.g. '2024-01-16'
    hindi: str        # Hindi rendering


class TranslationState(TypedDict, total=False):
    # ── Core ──────────────────────────────────────────────────
    source_text: str
    session_id: str
    source_lang: str          # default 'en'
    target_lang: str          # default 'hi'

    # ── NLP Enrichment (Direction 1) ──────────────────────────
    named_entities: list[NamedEntity]
    temporal_expressions: list[TemporalExpression]
    word_senses: dict[str, str]       # word → WordNet synset ID
    dependency_parse: dict            # UD parse structure
    pos_tags: list[dict]              # POS tags per token
    enrichment_context: str           # formatted string for LLM prompt

    # ── Knowledge Graph (Direction 2) ─────────────────────────
    kg_facts: list[dict]              # retrieved triples from KG
    hypernyms: dict[str, list[str]]   # word → list of hypernyms
    hyponyms: dict[str, list[str]]    # word → list of hyponyms
    kg_context: str                   # formatted KG facts for prompt

    # ── Memory ────────────────────────────────────────────────
    working_memory: dict[str, Any]
    episodic_hits: list[dict[str, Any]]
    semantic_hits: list[dict[str, Any]]
    source_embedding: list[float]

    # ── Translation ───────────────────────────────────────────
    translation: str
    iteration: int
    max_iterations: int

    # ── Evaluation ────────────────────────────────────────────
    scores: EvalScores
    reference_translations: list[str]
    passed: bool
    score_history: list[EvalScores]

    # ── Feedback ──────────────────────────────────────────────
    feedback: str

    # ── Output ────────────────────────────────────────────────
    final_translation: str
    final_scores: EvalScores

    # ── Back Translation (Direction 4) ────────────────────────
    is_back_translation: bool
    pivot_translation: str
    round_trip_score: float

    # ── Agent metadata (Direction 3) ──────────────────────────
    agent_trace: list[str]    # which agents were called
    messages: Annotated[list, add_messages]


print('✅ Expanded state schema defined')
print(f'   Fields: {len(TranslationState.__annotations__)}')

✅ Expanded state schema defined
   Fields: 33


## Step 4 — Memory Systems (Working, Episodic, Semantic)

In [ ]:
import json, time, uuid, numpy as np
from pathlib import Path
from typing import Any
from sqlitedict import SqliteDict
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

DATA_DIR = Path('/content/translation_data_v2')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Embedding Model ───────────────────────────────────────────────────────────
_embed_model = None

def get_embed_model():
    global _embed_model
    if _embed_model is None:
        print('Loading LaBSE embedding model...')
        _embed_model = SentenceTransformer('sentence-transformers/LaBSE')
        print('✅ LaBSE loaded')
    return _embed_model

def encode_text(text):
    model = get_embed_model()
    result = model.encode(text, normalize_embeddings=True)
    return result.tolist() if isinstance(text, str) else [r.tolist() for r in result]


# ── Working Memory ────────────────────────────────────────────────────────────
class WorkingMemory:
    def __init__(self, session_id: str):
        self.session_id = session_id
        self._store = {
            'session_id': session_id,
            'history': [],
            'domain_hint': None,
            'entity_cache': {},   # entity text → Hindi translation cache
            'created_at': time.time(),
        }

    def add_translation(self, source, translation, scores, entities=None):
        self._store['history'].append({
            'source': source, 'translation': translation,
            'scores': scores, 'timestamp': time.time(),
        })
        self._store['history'] = self._store['history'][-10:]
        # Cache entity translations for future sentences
        if entities:
            for ent in entities:
                if ent.get('hindi'):
                    self._store['entity_cache'][ent['text']] = ent['hindi']

    def get_entity_cache(self): return self._store.get('entity_cache', {})
    def get_recent(self, n=3): return self._store['history'][-n:]
    def set_domain(self, domain): self._store['domain_hint'] = domain
    def to_dict(self): return dict(self._store)


# ── Episodic Memory ───────────────────────────────────────────────────────────
class EpisodicMemory:
    def __init__(self):
        self._db = SqliteDict(str(DATA_DIR / 'episodic_v2.sqlite'), autocommit=True)

    def store(self, source, translation, scores, feedback,
              session_id, embedding, entities=None, temporal=None):
        ep_id = str(uuid.uuid4())
        self._db[ep_id] = json.dumps({
            'id': ep_id, 'source': source, 'translation': translation,
            'scores': scores, 'feedback': feedback, 'session_id': session_id,
            'embedding': embedding, 'entities': entities or [],
            'temporal': temporal or [], 'timestamp': time.time(),
        })
        return ep_id

    def retrieve_similar(self, query_embedding, top_k=3, min_score=0.5):
        q = np.array(query_embedding)
        results = []
        for raw in self._db.values():
            record = json.loads(raw)
            sim = float(np.dot(q, np.array(record['embedding'])))
            if sim >= min_score:
                results.append({**record, '_similarity': sim})
        return sorted(results, key=lambda x: x['_similarity'], reverse=True)[:top_k]

    def count(self): return len(self._db)


# ── Semantic Memory ───────────────────────────────────────────────────────────
class SemanticMemory:
    def __init__(self):
        self._client = chromadb.PersistentClient(
            path=str(DATA_DIR / 'chroma_v2'),
            settings=Settings(anonymized_telemetry=False))
        self._col = self._client.get_or_create_collection(
            name='hindi_knowledge_v2',
            metadata={'hnsw:space': 'cosine'})

    def add_knowledge(self, chunks, metadatas=None, ids=None):
        if not chunks: return
        embeddings = encode_text(chunks)
        ids = ids or [str(uuid.uuid4()) for _ in chunks]
        metadatas = metadatas or [{} for _ in chunks]
        self._col.upsert(documents=chunks, embeddings=embeddings,
                         metadatas=metadatas, ids=ids)

    def retrieve(self, query_embedding, top_k=5):
        if self._col.count() == 0: return []
        results = self._col.query(
            query_embeddings=[query_embedding],
            n_results=min(top_k, self._col.count()),
            include=['documents', 'metadatas', 'distances'])
        return [{'chunk': d, 'metadata': m, 'distance': dist}
                for d, m, dist in zip(results['documents'][0],
                                      results['metadatas'][0],
                                      results['distances'][0])]

    def count(self): return self._col.count()

    def seed_knowledge(self):
        if self._col.count() > 0:
            print(f'   Semantic memory already has {self._col.count()} chunks')
            return
        chunks = [
            'Hindi SOV word order: Subject-Object-Verb. Example: राम ने सेब खाया (Ram ate apple).',
            'Hindi postpositions follow nouns: में (in), पर (on), को (to/for), से (from/with), के लिए (for).',
            'Hindi verb agreement: masculine singular करता है, feminine singular करती है.',
            'Formal register: आप (you), कृपया (please), धन्यवाद (thank you), क्षमा करें (excuse me).',
            'Informal register: तुम (you), यार (friend), ठीक है (okay), क्या बात है (what is up).',
            'Ergative case: transitive verbs in perfective aspect, subject takes ने: राम ने खाया.',
            'English break the ice → Hindi बात की शुरुआत करना (to start conversation).',
            'English once in a blue moon → Hindi कभी-कभार (rarely).',
            'English raining cats and dogs → Hindi मूसलाधार बारिश (torrential rain).',
            'Transliterate technical terms: software=सॉफ़्टवेयर, computer=कंप्यूटर, internet=इंटरनेट.',
            'Medical: hospital=अस्पताल, doctor=डॉक्टर, medicine=दवाई, patient=मरीज़, surgery=शल्य चिकित्सा.',
            'Legal: court=न्यायालय, judge=न्यायाधीश, verdict=फ़ैसला, plaintiff=वादी, defendant=प्रतिवादी.',
            'Numbers: 1=एक 2=दो 3=तीन 4=चार 5=पाँच 10=दस 100=सौ 1000=हज़ार 100000=लाख.',
            'Time: morning=सुबह, afternoon=दोपहर, evening=शाम, night=रात, today=आज, tomorrow=कल.',
            'Named entity PERSON should be transliterated in Devanagari, not translated.',
            'Named entity ORG: use official Hindi name if exists, else transliterate.',
            'Named entity LOC: use standard Hindi place name from gazetteer.',
            'DATE entities should use Hindi numerals or Devanagari month names when appropriate.',
            'Plural feminine: लड़की→लड़कियाँ, औरत→औरतें. Plural masculine: लड़का→लड़के, आदमी→आदमी.',
            'Hindi compound postposition: के बारे में (about), के साथ (with), के बिना (without).',
        ]
        metadatas = [
            {'type': 'grammar'}, {'type': 'grammar'}, {'type': 'grammar'},
            {'type': 'register'}, {'type': 'register'}, {'type': 'grammar'},
            {'type': 'idiom'}, {'type': 'idiom'}, {'type': 'idiom'},
            {'type': 'technical'}, {'type': 'medical'}, {'type': 'legal'},
            {'type': 'vocabulary'}, {'type': 'vocabulary'},
            {'type': 'ner_rule'}, {'type': 'ner_rule'}, {'type': 'ner_rule'}, {'type': 'ner_rule'},
            {'type': 'grammar'}, {'type': 'grammar'},
        ]
        self.add_knowledge(chunks, metadatas)
        print(f'✅ Seeded {len(chunks)} knowledge chunks')


# ── Initialise all memory systems ─────────────────────────────────────────────
SESSION_ID      = str(uuid.uuid4())
working_memory  = WorkingMemory(SESSION_ID)
episodic_memory = EpisodicMemory()
semantic_memory = SemanticMemory()
semantic_memory.seed_knowledge()

print(f'\n✅ Memory systems ready  (session={SESSION_ID[:8]}...)')
print(f'   Episodic episodes : {episodic_memory.count()}')
print(f'   Semantic chunks   : {semantic_memory.count()}')

# Pre-load embedding model
_ = get_embed_model()

Loading LaBSE embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

✅ LaBSE loaded
✅ Seeded 20 knowledge chunks

✅ Memory systems ready  (session=44136a20...)
   Episodic episodes : 0
   Semantic chunks   : 20


## Step 5 — Direction 1: NLP Enrichment (NER, TIMEX, WordNet, DBpedia, VerbNet)

In [ ]:
import stanza
import re
from nltk.corpus import wordnet as wn
from nltk import pos_tag, word_tokenize

# Load Stanza pipelines (cached after first load)
_stanza_en = None
_stanza_hi = None

def get_stanza_en():
    global _stanza_en
    if _stanza_en is None:
        print('Loading Stanza English pipeline...')
        _stanza_en = stanza.Pipeline('en', processors='tokenize,pos,lemma,depparse,ner', verbose=False)
        print('✅ Stanza EN loaded')
    return _stanza_en

def get_stanza_hi():
    global _stanza_hi
    if _stanza_hi is None:
        print('Loading Stanza Hindi pipeline...')
        _stanza_hi = stanza.Pipeline('hi', processors='tokenize,pos,lemma,depparse', verbose=False)
        print('✅ Stanza HI loaded')
    return _stanza_hi


# ── Named Entity Recognition ──────────────────────────────────────────────────
def extract_named_entities(text: str) -> list[dict]:
    """
    Extract named entities using Stanza NER.
    Returns list of {text, type, start, end}
    """
    nlp = get_stanza_en()
    doc = nlp(text)
    entities = []
    for ent in doc.entities:
        entities.append({
            'text':  ent.text,
            'type':  ent.type,
            'start': ent.start_char,
            'end':   ent.end_char,
            'hindi': '',          # filled by KG lookup
            'dbpedia_uri': '',
        })
    return entities


# ── FIXED: extract_pos_and_parse ─────────────────────────────────────────────
# Stanza >= 1.5 renamed head_id → head
# Run this cell to overwrite the broken version

def extract_pos_and_parse(text: str) -> tuple[list[dict], dict]:
    """
    Returns POS tags and UD dependency parse structure.
    Compatible with Stanza >= 1.5
    """
    nlp = get_stanza_en()
    doc = nlp(text)
    pos_tags  = []
    dep_parse = {'words': [], 'dependencies': []}

    for sent in doc.sentences:
        for word in sent.words:
            pos_tags.append({
                'text':  word.text,
                'upos':  word.upos,
                'lemma': word.lemma,
            })
            dep_parse['words'].append(word.text)

            # Safely get head — handle both old (head_id) and new (head) API
            head = getattr(word, 'head', None) or getattr(word, 'head_id', None)

            dep_parse['dependencies'].append({
                'word':   word.text,
                'head':   head,          # fixed: was word.head_id
                'deprel': word.deprel,
                'lemma':  word.lemma,
            })

    return pos_tags, dep_parse




# ── TIMEX Temporal Expression Tagging ────────────────────────────────────────
TIMEX_PATTERNS = [
    # Absolute dates
    (r'\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b', 'DATE'),
    (r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s*\d{4}\b', 'DATE'),
    (r'\b\d{1,2}\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}\b', 'DATE'),
    # Relative expressions
    (r'\b(yesterday|today|tomorrow|next week|last week|next month|last month|next year|last year)\b', 'DATE'),
    (r'\b(\d+)\s+(days?|weeks?|months?|years?)\s+ago\b', 'DURATION'),
    (r'\bin\s+(\d+)\s+(days?|weeks?|months?|years?)\b', 'DURATION'),
    # Times
    (r'\b\d{1,2}:\d{2}\s*(AM|PM|am|pm)?\b', 'TIME'),
    (r'\b(morning|afternoon|evening|night|midnight|noon)\b', 'TIME'),
    # Recurring
    (r'\b(every\s+(day|week|month|year|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday))\b', 'SET'),
]

HINDI_MONTHS = {
    'January': 'जनवरी', 'February': 'फ़रवरी', 'March': 'मार्च',
    'April': 'अप्रैल', 'May': 'मई', 'June': 'जून',
    'July': 'जुलाई', 'August': 'अगस्त', 'September': 'सितंबर',
    'October': 'अक्टूबर', 'November': 'नवंबर', 'December': 'दिसंबर'
}

HINDI_TIME_WORDS = {
    'yesterday': 'कल', 'today': 'आज', 'tomorrow': 'कल',
    'morning': 'सुबह', 'afternoon': 'दोपहर', 'evening': 'शाम',
    'night': 'रात', 'midnight': 'आधी रात', 'noon': 'दोपहर',
    'next week': 'अगले हफ़्ते', 'last week': 'पिछले हफ़्ते',
    'next month': 'अगले महीने', 'last month': 'पिछले महीने',
    'next year': 'अगले साल', 'last year': 'पिछले साल',
}

def extract_temporal_expressions(text: str) -> list[dict]:
    """
    TIMEX-style temporal tagging using regex patterns.
    Returns list of {text, type, value, hindi}
    """
    expressions = []
    seen_spans = set()
    for pattern, timex_type in TIMEX_PATTERNS:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            span = (match.start(), match.end())
            if span not in seen_spans:
                seen_spans.add(span)
                expr_text = match.group(0).strip()
                hindi = HINDI_TIME_WORDS.get(expr_text.lower(), '')
                # Replace English month names with Hindi
                for eng, hin in HINDI_MONTHS.items():
                    expr_text_hi = expr_text.replace(eng, hin)
                expressions.append({
                    'text':  expr_text,
                    'type':  timex_type,
                    'value': expr_text,  # simplified; real TIMEX uses HeidelTime
                    'hindi': hindi or expr_text_hi if 'expr_text_hi' in dir() else '',
                })
    return expressions


# ── WordNet Sense Disambiguation ──────────────────────────────────────────────
def get_word_senses(text: str) -> dict[str, dict]:
    """
    Get WordNet synsets, hypernyms, hyponyms for content words.
    Returns {word: {synset, definition, hypernyms, hyponyms}}
    """
    tokens = word_tokenize(text.lower())
    tagged = pos_tag(tokens)
    senses = {}

    for word, pos in tagged:
        # Map NLTK POS to WordNet POS
        wn_pos = None
        if pos.startswith('NN'):   wn_pos = wn.NOUN
        elif pos.startswith('VB'): wn_pos = wn.VERB
        elif pos.startswith('JJ'): wn_pos = wn.ADJ
        elif pos.startswith('RB'): wn_pos = wn.ADV

        if wn_pos and len(word) > 3:
            synsets = wn.synsets(word, pos=wn_pos)
            if synsets:
                best = synsets[0]  # most common sense (simplified Lesk)
                hypernyms = [h.lemmas()[0].name().replace('_', ' ')
                             for h in best.hypernyms()[:3]]
                hyponyms  = [h.lemmas()[0].name().replace('_', ' ')
                             for h in best.hyponyms()[:3]]
                homonyms  = [s.definition() for s in synsets[1:3]]
                senses[word] = {
                    'synset':     best.name(),
                    'definition': best.definition(),
                    'hypernyms':  hypernyms,
                    'hyponyms':   hyponyms,
                    'homonyms':   homonyms,
                }
    return senses


# ── DBpedia Entity Linking (lightweight, no API key needed) ───────────────────
def link_to_dbpedia(entities: list[dict]) -> list[dict]:
    """
    Link named entities to DBpedia URIs via SPARQL endpoint.
    Falls back gracefully if endpoint is unavailable.
    """
    try:
        from SPARQLWrapper import SPARQLWrapper, JSON
        sparql = SPARQLWrapper('https://dbpedia.org/sparql')
        sparql.setTimeout(5)
        linked = []
        for ent in entities:
            try:
                query = f'''
                    SELECT ?uri ?label WHERE {{
                        ?uri rdfs:label "{ent['text']}"@en .
                        ?uri rdfs:label ?label .
                        FILTER (langMatches(lang(?label), "hi"))
                    }} LIMIT 1
                '''
                sparql.setQuery(query)
                sparql.setReturnFormat(JSON)
                results = sparql.query().convert()
                if results['results']['bindings']:
                    r = results['results']['bindings'][0]
                    ent = dict(ent)
                    ent['dbpedia_uri'] = r.get('uri', {}).get('value', '')
                    ent['hindi']       = r.get('label', {}).get('value', '')
            except Exception:
                pass
            linked.append(ent)
        return linked
    except Exception:
        return entities   # return unchanged if SPARQL unavailable


# ── Build enrichment context string for LLM prompt ───────────────────────────
def build_enrichment_context(
    entities: list[dict],
    temporal: list[dict],
    senses: dict,
    entity_cache: dict,
) -> str:
    lines = []

    if entities:
        lines.append('[Named entities detected]')
        for ent in entities:
            hindi = ent.get('hindi') or entity_cache.get(ent['text'], '')
            hint  = f' → translate as {hindi}' if hindi else ' → transliterate'
            lines.append(f'  • {ent["text"]}  [{ent["type"]}]{hint}')

    if temporal:
        lines.append('[Temporal expressions]')
        for t in temporal:
            hint = f' → {t["hindi"]}' if t.get('hindi') else ''
            lines.append(f'  • "{t["text"]}"  [{t["type"]}]{hint}')

    if senses:
        ambiguous = {w: s for w, s in senses.items() if len(s.get('homonyms', [])) > 0}
        if ambiguous:
            lines.append('[Ambiguous words — use correct sense]')
            for word, info in list(ambiguous.items())[:3]:
                lines.append(f'  • "{word}" = {info["definition"]}')
                if info['hypernyms']:
                    lines.append(f'    hypernym: {info["hypernyms"][0]}')

    return '\n'.join(lines)


print('✅ NLP enrichment functions defined')
print('   Tools: Stanza NER · TIMEX · WordNet · DBpedia')

✅ NLP enrichment functions defined
   Tools: Stanza NER · TIMEX · WordNet · DBpedia


## Step 6 — Direction 2: Knowledge Graph (NetworkX + Entity/Temporal/Wordnet nodes)

In [ ]:
import networkx as nx
import json
from pathlib import Path

KG_PATH = DATA_DIR / 'knowledge_graph.json'


class HindiKnowledgeGraph:
    """
    Directed knowledge graph using NetworkX.
    Node types: ENTITY, TIMEX, SYNSET, GRAMMAR_RULE, TRANSLATION
    Edge types: translates_to, is_a (hypernym), has_type (hyponym),
                similar_to (homonym), tagged_with, has_parse

    In production: replace with Neo4j using py2neo or neo4j driver.
    Cypher equivalent queries are shown as comments.
    """

    def __init__(self):
        self.G = nx.DiGraph()
        self._load_or_create()

    def _load_or_create(self):
        if KG_PATH.exists():
            data = json.loads(KG_PATH.read_text())
            self.G = nx.node_link_graph(data)
            print(f'   KG loaded: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges')
        else:
            self._seed_default_graph()
            self.save()

    def save(self):
        KG_PATH.write_text(json.dumps(nx.node_link_data(self.G)))

    # ── Node addition ────────────────────────────────────────────────────────
    def add_entity(self, text_en, text_hi, entity_type,
                   dbpedia_uri='', register='neutral', domain='general'):
        """
        Add bilingual entity pair.
        Neo4j equivalent:
          CREATE (:Entity {text:'text_en', lang:'en', type:'ORG'})
          CREATE (:Entity {text:'text_hi', lang:'hi', type:'ORG'})
          MATCH (a),(b) WHERE a.text=text_en AND b.text=text_hi
          CREATE (a)-[:translates_to]->(b)
        """
        node_en = f'EN:{text_en}'
        node_hi = f'HI:{text_hi}'
        self.G.add_node(node_en, text=text_en, lang='en', type=entity_type,
                        dbpedia_uri=dbpedia_uri)
        self.G.add_node(node_hi, text=text_hi, lang='hi', type=entity_type,
                        register=register, domain=domain)
        self.G.add_edge(node_en, node_hi, relation='translates_to')
        return node_en, node_hi

    def add_wordnet_synset(self, word, synset_id, definition, hypernyms, hyponyms):
        """
        Add WordNet synset with hypernym/hyponym edges.
        Neo4j equivalent:
          CREATE (:Synset {id:synset_id, word:word, definition:definition})
          FOREACH (h IN hypernyms |
            MERGE (:Synset {id:h})
            CREATE (s)-[:is_a]->(h_node))
        """
        node_id = f'SYN:{synset_id}'
        self.G.add_node(node_id, word=word, synset=synset_id,
                        definition=definition, node_type='SYNSET')
        for h in hypernyms:
            h_id = f'SYN:{h}'
            self.G.add_node(h_id, word=h, node_type='SYNSET')
            self.G.add_edge(node_id, h_id, relation='is_a')
        for h in hyponyms:
            h_id = f'SYN:{h}'
            self.G.add_node(h_id, word=h, node_type='SYNSET')
            self.G.add_edge(node_id, h_id, relation='has_type')

    def add_temporal_node(self, text, timex_type, iso_value, hindi):
        """
        Add temporal expression node.
        Neo4j: CREATE (:TIMEX {text:text, type:'DATE', value:iso_value})
        """
        node_id = f'TIMEX:{text}'
        self.G.add_node(node_id, text=text, timex_type=timex_type,
                        iso_value=iso_value, hindi=hindi, node_type='TIMEX')
        return node_id

    def add_grammar_rule(self, rule_id, rule_text, example_en, example_hi):
        node_id = f'RULE:{rule_id}'
        self.G.add_node(node_id, rule=rule_text, example_en=example_en,
                        example_hi=example_hi, node_type='GRAMMAR_RULE')

    # ── Queries ───────────────────────────────────────────────────────────────
    def get_hindi_translation(self, text_en: str) -> str:
        """
        MATCH (en:Entity {text:text_en})-[:translates_to]->(hi:Entity)
        RETURN hi.text
        """
        node_en = f'EN:{text_en}'
        if node_en in self.G:
            for _, target, data in self.G.out_edges(node_en, data=True):
                if data.get('relation') == 'translates_to':
                    return self.G.nodes[target].get('text', '')
        return ''

    def get_hypernyms(self, word: str) -> list[str]:
        """
        MATCH (:Synset {word:word})-[:is_a*1..2]->(h:Synset)
        RETURN h.word
        """
        results = []
        for node, data in self.G.nodes(data=True):
            if data.get('word') == word and data.get('node_type') == 'SYNSET':
                for _, target, edata in self.G.out_edges(node, data=True):
                    if edata.get('relation') == 'is_a':
                        results.append(self.G.nodes[target].get('word', ''))
        return results

    def get_temporal_hindi(self, text: str) -> str:
        node_id = f'TIMEX:{text}'
        if node_id in self.G:
            return self.G.nodes[node_id].get('hindi', '')
        return ''

    def query_facts_for_entity(self, text_en: str) -> list[dict]:
        """
        Get all facts about an entity from the KG.
        Neo4j: MATCH (e:Entity {text:text_en})-[r]->(n) RETURN type(r), n
        """
        node_en = f'EN:{text_en}'
        facts = []
        if node_en in self.G:
            for _, target, data in self.G.out_edges(node_en, data=True):
                target_data = self.G.nodes[target]
                facts.append({
                    'relation': data.get('relation'),
                    'target':   target_data.get('text', target),
                    'target_data': target_data,
                })
        return facts

    def build_context_for_sentence(
        self,
        entities: list[dict],
        temporal: list[dict],
        senses: dict,
    ) -> str:
        """Build KG-grounded context string for translation prompt."""
        lines = []

        # Entity translations from KG
        for ent in entities:
            hindi = self.get_hindi_translation(ent['text'])
            if hindi:
                ent['hindi'] = hindi
                lines.append(f'KG: {ent["text"]} [{ent["type"]}] → {hindi}')

        # Temporal expressions from KG
        for t in temporal:
            hindi = self.get_temporal_hindi(t['text'])
            if hindi:
                t['hindi'] = hindi
                lines.append(f'KG-TIMEX: "{t["text"]}" → {hindi}')

        # Hypernyms for disambiguation
        for word, info in list(senses.items())[:3]:
            hyp = self.get_hypernyms(word)
            if hyp:
                lines.append(f'KG-WordNet: "{word}" is_a {hyp[0]}')

        return '\n'.join(lines) if lines else ''

    # ── Default seed graph ───────────────────────────────────────────────────
    def _seed_default_graph(self):
        print('Seeding default knowledge graph...')

        # Named entities — persons
        self.add_entity('Narendra Modi',     'नरेंद्र मोदी',    'PERSON', domain='politics')
        self.add_entity('Amitabh Bachchan',  'अमिताभ बच्चन',   'PERSON', domain='entertainment')
        self.add_entity('Mahatma Gandhi',    'महात्मा गांधी',  'PERSON', domain='history')
        self.add_entity('Sachin Tendulkar',  'सचिन तेंदुलकर', 'PERSON', domain='sports')

        # Named entities — organisations
        self.add_entity('Supreme Court',            'सर्वोच्च न्यायालय',    'ORG', register='formal')
        self.add_entity('Reserve Bank of India',    'भारतीय रिज़र्व बैंक',  'ORG', register='formal')
        self.add_entity('Indian Railways',          'भारतीय रेलवे',          'ORG')
        self.add_entity('Parliament of India',      'भारतीय संसद',           'ORG', register='formal')
        self.add_entity('Election Commission',      'चुनाव आयोग',            'ORG')
        self.add_entity('ISRO',                     'इसरो',                  'ORG', domain='science')
        self.add_entity('Income Tax Department',    'आयकर विभाग',            'ORG', register='formal')

        # Named entities — locations
        self.add_entity('New Delhi',  'नई दिल्ली',  'LOC')
        self.add_entity('Mumbai',     'मुंबई',       'LOC')
        self.add_entity('Varanasi',   'वाराणसी',    'LOC')
        self.add_entity('Kolkata',    'कोलकाता',    'LOC')
        self.add_entity('Chennai',    'चेन्नई',      'LOC')
        self.add_entity('Bengaluru',  'बेंगलुरु',   'LOC')
        self.add_entity('Taj Mahal',  'ताज महल',    'LOC', domain='tourism')
        self.add_entity('Himalayas',  'हिमालय',     'LOC')
        self.add_entity('Ganges',     'गंगा',        'LOC')

        # Temporal expressions
        temporals = [
            ('today',      'DATE', 'PRESENT', 'आज'),
            ('yesterday',  'DATE', 'PAST',    'कल'),
            ('tomorrow',   'DATE', 'FUTURE',  'कल'),
            ('next week',  'DATE', 'FUTURE',  'अगले हफ़्ते'),
            ('last week',  'DATE', 'PAST',    'पिछले हफ़्ते'),
            ('next month', 'DATE', 'FUTURE',  'अगले महीने'),
            ('last month', 'DATE', 'PAST',    'पिछले महीने'),
            ('morning',    'TIME', 'PRESENT', 'सुबह'),
            ('evening',    'TIME', 'PRESENT', 'शाम'),
            ('midnight',   'TIME', 'PRESENT', 'आधी रात'),
        ]
        for text, ttype, value, hindi in temporals:
            self.add_temporal_node(text, ttype, value, hindi)

        # Grammar rules
        rules = [
            ('SOV', 'Hindi uses Subject-Object-Verb order', 'Ram eats apple', 'राम सेब खाता है'),
            ('ERG', 'Ergative: transitive perfective subject takes ने', 'Ram ate', 'राम ने खाया'),
            ('POST', 'Postpositions follow nouns: में पर को से के लिए', 'in the house', 'घर में'),
            ('AGR', 'Verb agrees with subject in gender/number', 'She goes', 'वह जाती है'),
            ('HON', 'Honorific आप takes plural verb even for one person', 'You are', 'आप हैं'),
        ]
        for rid, rule, ex_en, ex_hi in rules:
            self.add_grammar_rule(rid, rule, ex_en, ex_hi)

        # WordNet synsets for common ambiguous words
        common_synsets = [
            ('bank',   'bank.n.01', 'financial institution', ['institution'], ['savings bank', 'central bank']),
            ('bank',   'bank.n.09', 'sloping land beside water', ['geological formation'], []),
            ('run',    'run.v.01',  'move fast by using legs', ['travel'], ['sprint', 'jog']),
            ('light',  'light.n.01','electromagnetic radiation', ['radiation'], ['sunlight', 'moonlight']),
            ('fair',   'fair.a.01', 'free from favoritism', [], []),
            ('bright', 'bright.a.01','emitting or reflecting light readily', [], []),
        ]
        for word, sid, defn, hypers, hypos in common_synsets:
            self.add_wordnet_synset(word, sid, defn, hypers, hypos)

        print(f'✅ KG seeded: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges')


# Initialise knowledge graph
knowledge_graph = HindiKnowledgeGraph()
print(f'\n✅ Knowledge Graph ready')
print(f'   Nodes: {knowledge_graph.G.number_of_nodes()}')
print(f'   Edges: {knowledge_graph.G.number_of_edges()}')

Seeding default knowledge graph...
✅ KG seeded: 71 nodes, 30 edges

✅ Knowledge Graph ready
   Nodes: 71
   Edges: 30


## Step 7 — Evaluation Metrics

In [ ]:
from typing import Optional

METRIC_WEIGHTS = {
    'bleu': float(os.getenv('W_BLEU', '0.20')),
    'chrf': float(os.getenv('W_CHRF', '0.25')),
    'bert_score_f1': float(os.getenv('W_BERT', '0.30')),
    'comet': float(os.getenv('W_COMET', '0.25')),
}

def _devanagari_heuristic(text: str) -> float:
    if not text: return 0.0
    dev = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    ratio = min(dev / max(len(text), 1) * 1.2, 1.0)
    length = min(len(text.split()) / 3.0, 1.0)
    return round(ratio * 0.7 + length * 0.3, 4)

def compute_bleu(hypothesis, references):
    if not references: return 0.0
    try:
        from sacrebleu.metrics import BLEU
        return float(BLEU(effective_order=True).sentence_score(hypothesis, references).score)
    except: return 0.0

def compute_chrf(hypothesis, references):
    if not references: return 0.0
    try:
        from sacrebleu.metrics import CHRF
        return float(CHRF().sentence_score(hypothesis, references).score)
    except: return 0.0

def compute_bert_score(hypothesis, references):
    if not references: return _devanagari_heuristic(hypothesis)
    try:
        from bert_score import score as bscore
        _, _, F1 = bscore([hypothesis], [references[0]],
                          model_type='bert-base-multilingual-cased', lang='hi', verbose=False)
        return float(F1[0])
    except: return _devanagari_heuristic(hypothesis)

def compute_comet(source, hypothesis, references=None):
    try:
        from comet import download_model, load_from_checkpoint
        model = load_from_checkpoint(download_model('Unbabel/wmt22-comet-da'))
        data = [{'src': source, 'mt': hypothesis}]
        if references: data[0]['ref'] = references[0]
        raw = model.predict(data, batch_size=1, gpus=0).scores[0]
        return round(float((raw + 1.0) / 2.0), 4)
    except: return _devanagari_heuristic(hypothesis)

def compute_all_metrics(hypothesis, references, source):
    return {
        'bleu':          round(compute_bleu(hypothesis, references), 2),
        'chrf':          round(compute_chrf(hypothesis, references), 2),
        'bert_score_f1': round(compute_bert_score(hypothesis, references), 4),
        'comet':         round(compute_comet(source, hypothesis, references), 4),
    }

def aggregate_score(scores):
    bn = scores.get('bleu', 0.0) / 100.0
    cn = scores.get('chrf', 0.0) / 100.0
    bf = scores.get('bert_score_f1', 0.0)
    co = scores.get('comet', 0.0)
    has_refs = bn > 0 or cn > 0
    w = METRIC_WEIGHTS if has_refs else {'bleu': 0, 'chrf': 0, 'bert_score_f1': 0.5, 'comet': 0.5}
    return round(w['bleu']*bn + w['chrf']*cn + w['bert_score_f1']*bf + w['comet']*co, 4)

def format_score_table(history):
    if not history: return 'No history'
    rows = ['Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate']
    rows.append('─────┼────────┼────────┼───────────┼─────────┼──────────')
    for i, s in enumerate(history, 1):
        rows.append(f'  {i:2d} │ {s.get("bleu",0):6.2f} │ {s.get("chrf",0):6.2f} │'
                    f'   {s.get("bert_score_f1",0):.4f}  │ {s.get("comet",0):.4f}  │'
                    f'  {s.get("aggregate",0):.4f}')
    return '\n'.join(rows)

print('✅ Evaluation metrics defined')

✅ Evaluation metrics defined


## Step 8 — Direction 3: Specialist Agents with MCP-Style Tool Protocol

In [ ]:
import textwrap
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from functools import partial


# ── MCP Tool Registry ─────────────────────────────────────────────────────────
# Each tool has: name, description, function, input_schema
# This mimics the MCP tool protocol — agents expose capabilities as callable tools

MCP_TOOL_REGISTRY = {}

def register_tool(name, description, func, input_schema=None):
    MCP_TOOL_REGISTRY[name] = {
        'name': name,
        'description': description,
        'function': func,
        'input_schema': input_schema or {},
    }

def call_tool(tool_name: str, **kwargs):
    """MCP-style tool invocation."""
    if tool_name not in MCP_TOOL_REGISTRY:
        raise ValueError(f'Tool {tool_name} not registered')
    tool = MCP_TOOL_REGISTRY[tool_name]
    return tool['function'](**kwargs)


# ── LLM helper ────────────────────────────────────────────────────────────────
def get_llm(temperature=0.3):
    return ChatGroq(
        model='llama-3.3-70b-versatile',
        temperature=temperature,
        max_tokens=512,
        api_key=os.environ['GROQ_API_KEY'],
    )


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 1 — NER Agent
# ══════════════════════════════════════════════════════════════════════════════
def ner_agent_tool(text: str, use_dbpedia: bool = False) -> dict:
    """
    MCP Tool: extract_entities
    Runs Stanza NER, optionally links to DBpedia.
    """
    entities  = extract_named_entities(text)
    if use_dbpedia:
        entities = link_to_dbpedia(entities)
    # Enrich with KG translations
    for ent in entities:
        kg_hindi = knowledge_graph.get_hindi_translation(ent['text'])
        if kg_hindi:
            ent['hindi'] = kg_hindi
    return {'entities': entities, 'count': len(entities)}

register_tool(
    name='extract_entities',
    description='Extract named entities (PERSON, ORG, LOC, DATE) from English text using Stanza NER and enrich with KG translations',
    func=ner_agent_tool,
    input_schema={'text': 'string', 'use_dbpedia': 'boolean'},
)


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 2 — Temporal Agent
# ══════════════════════════════════════════════════════════════════════════════
def temporal_agent_tool(text: str) -> dict:
    """
    MCP Tool: extract_temporal_expressions
    TIMEX tagging + KG-grounded Hindi rendering.
    """
    expressions = extract_temporal_expressions(text)
    for expr in expressions:
        kg_hindi = knowledge_graph.get_temporal_hindi(expr['text'].lower())
        if kg_hindi:
            expr['hindi'] = kg_hindi
    return {'temporal_expressions': expressions, 'count': len(expressions)}

register_tool(
    name='extract_temporal_expressions',
    description='Detect and normalise temporal expressions (dates, times, durations) using TIMEX tagging',
    func=temporal_agent_tool,
    input_schema={'text': 'string'},
)


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 3 — Knowledge Graph Agent
# ══════════════════════════════════════════════════════════════════════════════
def kg_agent_tool(text: str, entities: list, temporal: list) -> dict:
    """
    MCP Tool: query_knowledge_graph
    Retrieves facts, hypernyms, translation hints from KG.
    """
    senses = get_word_senses(text)
    kg_context = knowledge_graph.build_context_for_sentence(entities, temporal, senses)
    hypernyms  = {w: knowledge_graph.get_hypernyms(w) for w in list(senses.keys())[:5]}
    facts = []
    for ent in entities:
        facts.extend(knowledge_graph.query_facts_for_entity(ent['text']))
    return {
        'kg_context': kg_context,
        'word_senses': senses,
        'hypernyms': hypernyms,
        'facts': facts,
    }

register_tool(
    name='query_knowledge_graph',
    description='Query the Hindi knowledge graph for entity translations, WordNet hypernyms, temporal Hindi mappings',
    func=kg_agent_tool,
    input_schema={'text': 'string', 'entities': 'list', 'temporal': 'list'},
)


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 4 — Translation Agent
# ══════════════════════════════════════════════════════════════════════════════
def translation_agent_tool(
    source: str, enrichment_context: str, kg_context: str,
    semantic_hits: list, episodic_hits: list, working_recent: list,
    feedback: str, iteration: int, max_iterations: int,
) -> dict:
    """
    MCP Tool: translate_to_hindi
    Core translation using LLM with all context injected.
    """
    # Build semantic knowledge section
    sem_section = ''
    if semantic_hits:
        chunks = '\n'.join(f'  • {h["chunk"]}' for h in semantic_hits[:4])
        sem_section = f'\n\n[Semantic knowledge]\n{chunks}'

    # Build episodic examples
    epi_section = ''
    if episodic_hits:
        lines = [f'  EN: {e["source"]}\n  HI: {e["translation"]}' for e in episodic_hits[:2]]
        epi_section = '\n\n[Similar past translations]\n' + '\n---\n'.join(lines)

    # Working memory context
    wm_section = ''
    if working_recent:
        lines = [f'  EN: {r["source"]}\n  HI: {r["translation"]}' for r in working_recent[-3:]]
        wm_section = '\n\n[Recent session translations]\n' + '\n'.join(lines)

    # NLP enrichment
    enr_section = f'\n\n[NLP enrichment — use these hints]\n{enrichment_context}' if enrichment_context else ''

    # KG grounded facts
    kg_section = f'\n\n[Knowledge graph facts]\n{kg_context}' if kg_context else ''

    # Feedback from previous attempt
    fb_section = f'\n\n[Previous attempt critique — fix these issues]\n{feedback}' if (iteration > 0 and feedback) else ''

    system_prompt = textwrap.dedent(f"""
        You are an expert English-to-Hindi translator with deep knowledge of:
        - Standard Hindi (Manak Hindi / खड़ी बोली) and Devanagari script
        - Register matching (formal आप vs informal तुम)
        - Hindi SOV word order and postposition system
        - Ergative case marking with ने
        - Cultural equivalence over literal translation

        RULES:
        1. Output ONLY the Hindi translation in Devanagari — no romanisation, no explanation.
        2. Named entities: use the exact Hindi form provided in NLP enrichment hints.
        3. Dates/times: use the Hindi rendering provided in temporal hints.
        4. Ambiguous words: use the sense indicated by WordNet hints.
        5. Preserve formal/informal register of the source.
        6. Transliterate technical terms with no Hindi equivalent.
        7. This is iteration {iteration + 1} of {max_iterations}.
        {sem_section}{epi_section}{wm_section}{enr_section}{kg_section}{fb_section}
    """).strip()

    llm = get_llm(temperature=0.2 + 0.05 * iteration)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f'Translate to Hindi:\n\n{source}'),
    ])
    translation = response.content.strip()
    return {'translation': translation}

register_tool(
    name='translate_to_hindi',
    description='Translate English text to Hindi using LLM with enriched context from NER, KG, memory',
    func=translation_agent_tool,
    input_schema={'source': 'string', 'enrichment_context': 'string', 'kg_context': 'string'},
)


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 5 — Evaluator Agent
# ══════════════════════════════════════════════════════════════════════════════
def evaluator_agent_tool(
    source: str, hypothesis: str,
    references: list, iteration: int, max_iterations: int,
) -> dict:
    """
    MCP Tool: evaluate_translation
    Runs all four metrics and decides pass/retry.
    """
    scores   = compute_all_metrics(hypothesis, references, source)
    agg      = aggregate_score(scores)
    scores['aggregate'] = agg
    threshold = float(os.getenv('PASS_THRESHOLD', '0.60'))
    passed    = agg >= threshold or iteration >= max_iterations
    return {'scores': scores, 'passed': passed, 'aggregate': agg}

register_tool(
    name='evaluate_translation',
    description='Score a Hindi translation using BLEU, chrF, BERTScore, COMET and decide pass/retry',
    func=evaluator_agent_tool,
    input_schema={'source': 'string', 'hypothesis': 'string', 'references': 'list'},
)


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 6 — Feedback Agent
# ══════════════════════════════════════════════════════════════════════════════
def feedback_agent_tool(
    source: str, translation: str, scores: dict, passed: bool,
) -> dict:
    """
    MCP Tool: generate_feedback
    Produces LLM critique targeting specific linguistic errors.
    """
    if passed:
        feedback = (f"Translation accepted. aggregate={scores.get('aggregate',0):.3f} "
                    f"BLEU={scores.get('bleu','N/A')} BERTScore={scores.get('bert_score_f1',0):.3f}")
        return {'feedback': feedback}

    score_str = (f"BLEU:{scores.get('bleu',0):.1f} chrF:{scores.get('chrf',0):.1f} "
                 f"BERT:{scores.get('bert_score_f1',0):.3f} agg:{scores.get('aggregate',0):.3f}")

    critic_prompt = textwrap.dedent(f"""
        You are a professional Hindi language quality assessor.
        Analyse the translation and provide 2-4 specific actionable issues.

        English: {source}
        Hindi  : {translation}
        Scores : {score_str}

        Check specifically for:
        - Named entity mistranslation (should be transliterated)
        - Temporal expression errors
        - Wrong postposition (में/पर/को/से/के लिए)
        - Gender/number agreement error
        - Ergative case missing ने
        - Wrong register (too formal/informal)
        - Literal translation of idioms
        - Missing or wrong diacritics

        Format:
        Issue 1: <specific problem + correction>
        Issue 2: <specific problem + correction>
        Overall: <one sentence priority fix>
    """).strip()

    llm = get_llm(temperature=0.1)
    feedback = llm.invoke([HumanMessage(content=critic_prompt)]).content.strip()
    return {'feedback': feedback}

register_tool(
    name='generate_feedback',
    description='Generate targeted linguistic critique for a failed translation to guide retry attempts',
    func=feedback_agent_tool,
    input_schema={'source': 'string', 'translation': 'string', 'scores': 'dict'},
)


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 7 — Memory Agent
# ══════════════════════════════════════════════════════════════════════════════
def memory_agent_tool(
    source: str, translation: str, scores: dict,
    feedback: str, session_id: str, embedding: list,
    entities: list, temporal: list,
) -> dict:
    """
    MCP Tool: update_memory
    Writes episode to all 3 tiers and updates the knowledge graph.
    """
    # Working memory
    working_memory.add_translation(source, translation, scores, entities)

    # Episodic memory
    if embedding:
        episodic_memory.store(source, translation, scores, feedback,
                              session_id, embedding, entities, temporal)

    # Semantic memory + KG — only for high quality
    agg = scores.get('aggregate', 0)
    if agg >= 0.70 and embedding:
        chunk = f'English: {source}\nHindi: {translation}'
        semantic_memory.add_knowledge(
            [chunk],
            [{'type': 'translation_pair', 'score': round(agg, 4)}]
        )
        # Update KG with new entity translations
        for ent in entities:
            if ent.get('hindi') and ent.get('type'):
                knowledge_graph.add_entity(
                    ent['text'], ent['hindi'], ent['type']
                )
        knowledge_graph.save()

    return {
        'working_entries': len(working_memory.to_dict()['history']),
        'episodic_count':  episodic_memory.count(),
        'semantic_chunks': semantic_memory.count(),
        'kg_updated':      agg >= 0.70,
    }

register_tool(
    name='update_memory',
    description='Persist completed translation episode to working, episodic, semantic memory and knowledge graph',
    func=memory_agent_tool,
    input_schema={'source': 'string', 'translation': 'string', 'scores': 'dict'},
)


print(f'✅ {len(MCP_TOOL_REGISTRY)} MCP tools registered:')
for name, tool in MCP_TOOL_REGISTRY.items():
    print(f'   • {name}: {tool["description"][:60]}...')

✅ 7 MCP tools registered:
   • extract_entities: Extract named entities (PERSON, ORG, LOC, DATE) from English...
   • extract_temporal_expressions: Detect and normalise temporal expressions (dates, times, dur...
   • query_knowledge_graph: Query the Hindi knowledge graph for entity translations, Wor...
   • translate_to_hindi: Translate English text to Hindi using LLM with enriched cont...
   • evaluate_translation: Score a Hindi translation using BLEU, chrF, BERTScore, COMET...
   • generate_feedback: Generate targeted linguistic critique for a failed translati...
   • update_memory: Persist completed translation episode to working, episodic, ...


## Step 9 — LangGraph Nodes (using MCP agents internally)

In [ ]:
# ── Node 1: Input ─────────────────────────────────────────────────────────────
def input_node(state: TranslationState) -> dict:
    source = state.get('source_text', '').strip()
    if not source: raise ValueError('source_text must not be empty')
    print(f"\n{'='*65}")
    print(f"[Node 1 — Input] '{source[:70]}{'...' if len(source)>70 else ''}'")
    return {
        'source_text': source, 'source_lang': state.get('source_lang', 'en'),
        'target_lang': state.get('target_lang', 'hi'),
        'iteration': 0, 'max_iterations': state.get('max_iterations', 3),
        'score_history': [], 'passed': False, 'translation': '',
        'final_translation': '', 'agent_trace': ['input_node'],
    }


# ── Node 2: NLP Enrichment (Direction 1) ──────────────────────────────────────
def enrichment_node(state: TranslationState) -> dict:
    source = state['source_text']

    # Call NER agent tool
    ner_result  = call_tool('extract_entities', text=source, use_dbpedia=False)
    entities    = ner_result['entities']

    # Call Temporal agent tool
    temp_result = call_tool('extract_temporal_expressions', text=source)
    temporal    = temp_result['temporal_expressions']

    # POS tags + dependency parse
    pos_tags, dep_parse = extract_pos_and_parse(source)

    # Build enrichment context string
    entity_cache    = working_memory.get_entity_cache()
    senses          = get_word_senses(source)
    enr_context     = build_enrichment_context(entities, temporal, senses, entity_cache)

    print(f'[Node 2 — Enrichment] entities={len(entities)} temporal={len(temporal)} senses={len(senses)}')

    trace = list(state.get('agent_trace', []))
    trace.append('enrichment_node')
    return {
        'named_entities': entities,
        'temporal_expressions': temporal,
        'word_senses': {w: s['synset'] for w, s in senses.items()},
        'pos_tags': pos_tags,
        'dependency_parse': dep_parse,
        'enrichment_context': enr_context,
        'agent_trace': trace,
    }


# ── Node 3: Knowledge Graph Query (Direction 2) ───────────────────────────────
def kg_node(state: TranslationState) -> dict:
    entities = state.get('named_entities', [])
    temporal = state.get('temporal_expressions', [])
    senses   = get_word_senses(state['source_text'])

    kg_result  = call_tool('query_knowledge_graph',
                           text=state['source_text'],
                           entities=entities, temporal=temporal)

    print(f'[Node 3 — KG] facts={len(kg_result["facts"])} kg_context_len={len(kg_result["kg_context"])}')

    trace = list(state.get('agent_trace', []))
    trace.append('kg_node')
    return {
        'kg_facts':    kg_result['facts'],
        'hypernyms':   kg_result['hypernyms'],
        'kg_context':  kg_result['kg_context'],
        'agent_trace': trace,
    }


# ── Node 4: Embedding + Memory Retrieval ─────────────────────────────────────
def embedding_node(state: TranslationState) -> dict:
    source    = state['source_text']
    embedding = encode_text(source)

    episodic_hits = episodic_memory.retrieve_similar(embedding, top_k=3, min_score=0.6)
    semantic_hits = semantic_memory.retrieve(embedding, top_k=5)

    print(f'[Node 4 — Embedding] dim={len(embedding)} epi={len(episodic_hits)} sem={len(semantic_hits)}')

    trace = list(state.get('agent_trace', []))
    trace.append('embedding_node')
    return {
        'source_embedding': embedding,
        'episodic_hits':    episodic_hits,
        'semantic_hits':    semantic_hits,
        'working_memory':   working_memory.to_dict(),
        'agent_trace':      trace,
    }


# ── Node 5: Translation (calls Translation Agent) ────────────────────────────
def translation_node(state: TranslationState) -> dict:
    result = call_tool(
        'translate_to_hindi',
        source             = state['source_text'],
        enrichment_context = state.get('enrichment_context', ''),
        kg_context         = state.get('kg_context', ''),
        semantic_hits      = state.get('semantic_hits', []),
        episodic_hits      = state.get('episodic_hits', []),
        working_recent     = working_memory.get_recent(3),
        feedback           = state.get('feedback', ''),
        iteration          = state.get('iteration', 0),
        max_iterations     = state.get('max_iterations', 3),
    )
    translation = result['translation']
    iteration   = state.get('iteration', 0) + 1

    print(f"[Node 5 — Translation] iter={iteration}: '{translation[:60]}{'...' if len(translation)>60 else ''}'")

    trace = list(state.get('agent_trace', []))
    trace.append(f'translation_node_iter{iteration}')
    return {'translation': translation, 'iteration': iteration, 'agent_trace': trace}


# ── Node 6: Evaluator (calls Evaluator Agent) ─────────────────────────────────
def evaluator_node(state: TranslationState) -> dict:
    result = call_tool(
        'evaluate_translation',
        source        = state['source_text'],
        hypothesis    = state['translation'],
        references    = state.get('reference_translations', []),
        iteration     = state.get('iteration', 1),
        max_iterations = state.get('max_iterations', 3),
    )
    scores  = result['scores']
    passed  = result['passed']
    history = list(state.get('score_history', []))
    history.append(dict(scores))

    status = '✅ PASSED' if passed else '🔄 RETRY'
    print(f"[Node 6 — Evaluator] {status} agg={scores.get('aggregate',0):.3f} "
          f"BLEU={scores.get('bleu',0):.1f} BERT={scores.get('bert_score_f1',0):.3f}")

    trace = list(state.get('agent_trace', []))
    trace.append('evaluator_node')
    return {'scores': scores, 'passed': passed, 'score_history': history, 'agent_trace': trace}


# ── Node 7: Feedback (calls Feedback Agent) ───────────────────────────────────
def feedback_node(state: TranslationState) -> dict:
    result = call_tool(
        'generate_feedback',
        source      = state['source_text'],
        translation = state['translation'],
        scores      = state.get('scores', {}),
        passed      = state.get('passed', False),
    )
    feedback = result['feedback']
    print(f'[Node 7 — Feedback] {feedback[:80]}...' if len(feedback) > 80 else f'[Node 7 — Feedback] {feedback}')

    trace = list(state.get('agent_trace', []))
    trace.append('feedback_node')
    return {'feedback': feedback, 'agent_trace': trace}


# ── Node 8: Memory Update (calls Memory Agent) ───────────────────────────────
def memory_update_node(state: TranslationState) -> dict:
    result = call_tool(
        'update_memory',
        source      = state['source_text'],
        translation = state['translation'],
        scores      = state.get('scores', {}),
        feedback    = state.get('feedback', ''),
        session_id  = state.get('session_id', SESSION_ID),
        embedding   = state.get('source_embedding', []),
        entities    = state.get('named_entities', []),
        temporal    = state.get('temporal_expressions', []),
    )
    print(f"[Node 8 — Memory] episodic={result['episodic_count']} "
          f"semantic={result['semantic_chunks']} kg_updated={result['kg_updated']}")

    trace = list(state.get('agent_trace', []))
    trace.append('memory_update_node')
    return {'working_memory': working_memory.to_dict(), 'agent_trace': trace}


# ── Node 9: Output ────────────────────────────────────────────────────────────
def output_node(state: TranslationState) -> dict:
    translation   = state['translation']
    scores        = state.get('scores', {})
    score_history = state.get('score_history', [])
    report        = format_score_table(score_history)

    print(f"[Node 9 — Output] '{translation}'")
    print(f'\n{report}')
    print(f'\nAgent trace: {" → ".join(state.get("agent_trace", []))}')

    return {
        'final_translation': translation,
        'final_scores': scores,
    }


# ── Conditional edge ──────────────────────────────────────────────────────────
def should_retry(state: TranslationState) -> str:
    return 'proceed' if state.get('passed', False) else 'retry'


print('✅ All 9 LangGraph nodes defined')

✅ All 9 LangGraph nodes defined


## Step 10 — Build & Compile the LangGraph

In [ ]:
from langgraph.graph import StateGraph, END

def build_graph():
    graph = StateGraph(TranslationState)

    # Add all nodes
    graph.add_node('input_node',         input_node)
    graph.add_node('enrichment_node',    enrichment_node)   # Direction 1
    graph.add_node('kg_node',            kg_node)           # Direction 2
    graph.add_node('embedding_node',     embedding_node)
    graph.add_node('translation_node',   translation_node)  # Direction 3
    graph.add_node('evaluator_node',     evaluator_node)    # Direction 3
    graph.add_node('feedback_node',      feedback_node)     # Direction 3
    graph.add_node('memory_update_node', memory_update_node)# Direction 3
    graph.add_node('output_node',        output_node)

    # Linear flow: input → enrich → KG → embed → translate → evaluate
    graph.add_edge('input_node',       'enrichment_node')
    graph.add_edge('enrichment_node',  'kg_node')
    graph.add_edge('kg_node',          'embedding_node')
    graph.add_edge('embedding_node',   'translation_node')
    graph.add_edge('translation_node', 'evaluator_node')

    # Conditional retry loop
    graph.add_conditional_edges(
        'evaluator_node', should_retry,
        {'retry': 'translation_node', 'proceed': 'feedback_node'}
    )

    # Continue to memory and output
    graph.add_edge('feedback_node',      'memory_update_node')
    graph.add_edge('memory_update_node', 'output_node')
    graph.add_edge('output_node',        END)

    graph.set_entry_point('input_node')
    return graph.compile()


app = build_graph()
print('✅ LangGraph V2 compiled')
print('   Flow: input → enrich → KG → embed → translate → evaluate')
print('         ↑ (retry) ←─────────────────────────────────────────┘')
print('         → feedback → memory_update → output')

✅ LangGraph V2 compiled
   Flow: input → enrich → KG → embed → translate → evaluate
         ↑ (retry) ←─────────────────────────────────────────┘
         → feedback → memory_update → output


## Step 11 — High-Level translate() Function

In [ ]:
def translate(
    source_text: str,
    reference_translations: list = None,
    max_iterations: int = 3,
    pass_threshold: float = 0.60,
    source_lang: str = 'en',
    target_lang: str = 'hi',
) -> dict:
    """
    Translate English → Hindi using the full V2 agentic pipeline.
    Includes NLP enrichment, KG grounding, memory, evaluation, retry.
    """
    os.environ['PASS_THRESHOLD'] = str(pass_threshold)

    initial_state = {
        'source_text':            source_text,
        'session_id':             SESSION_ID,
        'source_lang':            source_lang,
        'target_lang':            target_lang,
        'reference_translations': reference_translations or [],
        'max_iterations':         max_iterations,
    }

    final_state = app.invoke(initial_state)

    return {
        'final_translation':    final_state.get('final_translation', ''),
        'final_scores':         final_state.get('final_scores', {}),
        'score_history':        final_state.get('score_history', []),
        'feedback':             final_state.get('feedback', ''),
        'iterations':           final_state.get('iteration', 0),
        'named_entities':       final_state.get('named_entities', []),
        'temporal_expressions': final_state.get('temporal_expressions', []),
        'agent_trace':          final_state.get('agent_trace', []),
        'kg_facts':             final_state.get('kg_facts', []),
    }


print('✅ translate() ready')

✅ translate() ready


## Step 12 — Run Tests

In [ ]:
# Test 1 — Entity-rich sentence
r1 = translate(
    source_text='Narendra Modi visited the Supreme Court in New Delhi yesterday.',
    max_iterations=3, pass_threshold=0.55,
)
print('\n' + '='*65)
print(f"ENGLISH  : Narendra Modi visited the Supreme Court in New Delhi yesterday.")
print(f"HINDI    : {r1['final_translation']}")
print(f"ENTITIES : {[(e['text'], e['type'], e.get('hindi','')) for e in r1['named_entities']]}")
print(f"TEMPORAL : {[(t['text'], t.get('hindi','')) for t in r1['temporal_expressions']]}")
print(f"KG FACTS : {len(r1['kg_facts'])} facts retrieved")
print(f"SCORE    : {r1['final_scores'].get('aggregate',0):.3f}")


[Node 1 — Input] 'Narendra Modi visited the Supreme Court in New Delhi yesterday.'
Loading Stanza English pipeline...
✅ Stanza EN loaded
[Node 2 — Enrichment] entities=4 temporal=1 senses=5
[Node 3 — KG] facts=2 kg_context_len=100
[Node 4 — Embedding] dim=768 epi=0 sem=5
[Node 5 — Translation] iter=1: 'नरेंद्र मोदी ने कल नई दिल्ली में सुप्रीम कोर्ट का दौरा किया।'
[Node 6 — Evaluator] ✅ PASSED agg=0.986 BLEU=0.0 BERT=0.986
[Node 7 — Feedback] Translation accepted. aggregate=0.986 BLEU=0.0 BERTScore=0.986
[Node 8 — Memory] episodic=1 semantic=21 kg_updated=True
[Node 9 — Output] 'नरेंद्र मोदी ने कल नई दिल्ली में सुप्रीम कोर्ट का दौरा किया।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │   0.00 │   0.00 │   0.9860  │ 0.9860  │  0.9860

Agent trace: input_node → enrichment_node → kg_node → embedding_node → translation_node_iter1 → evaluator_node → feedback_node → memory_update_node

ENGLISH  : Narendra Modi visited

In [ ]:
# Test 2 — Temporal expression + ambiguous word
r2 = translate(
    source_text='The Reserve Bank of India will announce its decision next month at morning.',
    reference_translations=['भारतीय रिज़र्व बैंक अगले महीने सुबह अपना निर्णय घोषित करेगा।'],
    max_iterations=3, pass_threshold=0.60,
)
print('\n' + '='*65)
print(f"ENGLISH  : The Reserve Bank of India will announce its decision next month at morning.")
print(f"HINDI    : {r2['final_translation']}")
print(f"SCORE    : {r2['final_scores'].get('aggregate',0):.3f}")
print(f"ITERS    : {r2['iterations']}")
print(f'\n{format_score_table(r2["score_history"])}')


[Node 1 — Input] 'The Reserve Bank of India will announce its decision next month at mor...'
[Node 2 — Enrichment] entities=3 temporal=2 senses=8
[Node 3 — KG] facts=0 kg_context_len=98
[Node 4 — Embedding] dim=768 epi=0 sem=5
[Node 5 — Translation] iter=1: 'रिज़र्व बैंक ऑफ़ इंडिया अगले महीने सुबह अपना निर्णय घोषित कर...'


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 6 — Evaluator] ✅ PASSED agg=0.872 BLEU=63.2 BERT=0.968
[Node 7 — Feedback] Translation accepted. aggregate=0.872 BLEU=63.16 BERTScore=0.968
[Node 8 — Memory] episodic=2 semantic=22 kg_updated=True
[Node 9 — Output] 'रिज़र्व बैंक ऑफ़ इंडिया अगले महीने सुबह अपना निर्णय घोषित करेगा।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  63.16 │  81.97 │   0.9682  │ 1.0000  │  0.8717

Agent trace: input_node → enrichment_node → kg_node → embedding_node → translation_node_iter1 → evaluator_node → feedback_node → memory_update_node

ENGLISH  : The Reserve Bank of India will announce its decision next month at morning.
HINDI    : रिज़र्व बैंक ऑफ़ इंडिया अगले महीने सुबह अपना निर्णय घोषित करेगा।
SCORE    : 0.872
ITERS    : 1

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  63.16 │  81.97 │   0.9682  │ 1.0000  │  0.8717


In [ ]:
# Test 3 — Idiom + WordNet disambiguation
r3 = translate(
    source_text="Sachin Tendulkar broke the ice at the press conference last evening.",
    max_iterations=3, pass_threshold=0.55,
)
print('\n' + '='*65)
print(f"ENGLISH : Sachin Tendulkar broke the ice at the press conference last evening.")
print(f"HINDI   : {r3['final_translation']}")
print(f"SCORE   : {r3['final_scores'].get('aggregate',0):.3f}")
print(f"TRACE   : {' → '.join(r3['agent_trace'])}")


[Node 1 — Input] 'Sachin Tendulkar broke the ice at the press conference last evening.'
[Node 2 — Enrichment] entities=2 temporal=1 senses=5
[Node 3 — KG] facts=1 kg_context_len=71
[Node 4 — Embedding] dim=768 epi=0 sem=5
[Node 5 — Translation] iter=1: 'सचिन तेंदुलकर ने कल शाम प्रेस कॉन्फ़्रेंस में बर्फ तोड़ दी।'
[Node 6 — Evaluator] ✅ PASSED agg=0.998 BLEU=0.0 BERT=0.998
[Node 7 — Feedback] Translation accepted. aggregate=0.998 BLEU=0.0 BERTScore=0.998
[Node 8 — Memory] episodic=3 semantic=23 kg_updated=True
[Node 9 — Output] 'सचिन तेंदुलकर ने कल शाम प्रेस कॉन्फ़्रेंस में बर्फ तोड़ दी।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │   0.00 │   0.00 │   0.9976  │ 0.9976  │  0.9976

Agent trace: input_node → enrichment_node → kg_node → embedding_node → translation_node_iter1 → evaluator_node → feedback_node → memory_update_node

ENGLISH : Sachin Tendulkar broke the ice at the press conference last evening.
HINDI

## Step 13 — Direction 4: Back Translation for Low-Resource Languages

Generates synthetic parallel data for Bhojpuri, Maithili, Awadhi etc.
Uses your existing translation pipeline as the engine.

In [ ]:
import numpy as np

# ── Round-trip similarity checker ────────────────────────────────────────────
def compute_round_trip_similarity(original: str, back_translated: str) -> float:
    """
    Cosine similarity between original and back-translated sentence embeddings.
    High similarity = the round trip preserved meaning = good translation pair.
    """
    emb_orig = np.array(encode_text(original))
    emb_back = np.array(encode_text(back_translated))
    return float(np.dot(emb_orig, emb_back))


# ── Low-resource language config ──────────────────────────────────────────────
LOW_RESOURCE_CONFIG = {
    'bho': {
        'name': 'Bhojpuri',
        'script': 'Devanagari',
        'sample_sentences': [
            'हम बाजार जाइत बानी।',      # I am going to the market
            'ऊ लोग खाना खात बाड़न।',     # They are eating food
            'रउरा के नाम का बा?',         # What is your name?
            'आज मौसम बहुत नीमन बा।',     # The weather is very good today
            'हमार घर इहाँ बा।',           # My house is here
        ]
    },
    'mai': {
        'name': 'Maithili',
        'script': 'Devanagari',
        'sample_sentences': [
            'हम बजार जाइत छी।',          # I am going to market
            'अहाँक नाम की अछि?',          # What is your name?
            'आइ मौसम बड्ड नीक अछि।',     # Weather is very good today
            'हमर घर एतय अछि।',            # My house is here
        ]
    },
    'awa': {
        'name': 'Awadhi',
        'script': 'Devanagari',
        'sample_sentences': [
            'हम बाजार जात हन।',           # I go to market
            'तोहार नाव का ह?',             # What is your name?
            'आज मौसम बहुत नीक ह।',        # Weather is very good today
            'हमार घर इहाँ ह।',             # My house is here
        ]
    },
}


# ── Back Translation Pipeline ─────────────────────────────────────────────────
def back_translation_pipeline(
    monolingual_sentences: list[str],
    source_lang_name: str,
    quality_threshold: float = 0.55,
    max_pairs: int = 50,
) -> list[dict]:
    """
    Generate synthetic parallel pairs for low-resource language.

    Process:
    1. Forward: low-resource sentence → Hindi pivot (via LLM)
    2. Back: Hindi pivot → approximate low-resource reconstruction
    3. Quality filter: cosine similarity of original vs back-translated
    4. High-quality pairs saved as (low-resource, Hindi) training data

    Args:
        monolingual_sentences: list of sentences in the low-resource language
        source_lang_name: display name e.g. 'Bhojpuri'
        quality_threshold: min round-trip similarity to keep pair
        max_pairs: maximum synthetic pairs to generate
    """
    print(f'\nBack-translation pipeline for {source_lang_name}')
    print(f'Input sentences: {len(monolingual_sentences)}')
    print(f'Quality threshold: {quality_threshold}')
    print('='*50)

    synthetic_pairs = []
    llm = get_llm(temperature=0.2)

    for i, sentence in enumerate(monolingual_sentences[:max_pairs]):
        print(f'\nProcessing [{i+1}/{min(len(monolingual_sentences), max_pairs)}]: {sentence}')

        try:
            # ── Step 1: Forward translation (low-resource → Hindi) ────────────
            forward_prompt = textwrap.dedent(f"""
                You are an expert translator of Indic languages.
                Translate the following {source_lang_name} sentence to standard Hindi (Manak Hindi).
                Output ONLY the Hindi translation in Devanagari. No explanation.

                {source_lang_name}: {sentence}
                Hindi:
            """).strip()

            forward_response = llm.invoke([HumanMessage(content=forward_prompt)])
            hindi_pivot = forward_response.content.strip()
            print(f'  → Hindi pivot: {hindi_pivot}')

            # ── Step 2: Back translation (Hindi → low-resource) ───────────────
            back_prompt = textwrap.dedent(f"""
                You are an expert translator of Indic languages.
                Translate the following Hindi sentence back to {source_lang_name}.
                Output ONLY the {source_lang_name} translation in Devanagari. No explanation.

                Hindi: {hindi_pivot}
                {source_lang_name}:
            """).strip()

            back_response = llm.invoke([HumanMessage(content=back_prompt)])
            back_translated = back_response.content.strip()
            print(f'  → Back-translated: {back_translated}')

            # ── Step 3: Round-trip quality check ─────────────────────────────
            rt_score = compute_round_trip_similarity(sentence, back_translated)
            print(f'  → Round-trip similarity: {rt_score:.3f}', end='')

            if rt_score >= quality_threshold:
                print(' ✅ KEPT')
                synthetic_pairs.append({
                    'source':            sentence,       # original low-resource
                    'pivot_translation': hindi_pivot,    # synthetic Hindi
                    'back_translated':   back_translated,
                    'rt_score':          round(rt_score, 4),
                    'source_lang':       source_lang_name,
                    'lang_pair':         f'{source_lang_name}→Hindi',
                    'instruction':       'Translate to Hindi:',
                    'input':             sentence,
                    'output':            hindi_pivot,
                })

                # Add to semantic memory as knowledge
                chunk = (f'{source_lang_name}: {sentence}\nHindi: {hindi_pivot}')
                semantic_memory.add_knowledge(
                    [chunk],
                    [{'type': 'back_translation', 'lang': source_lang_name, 'rt_score': rt_score}]
                )
            else:
                print(f' ❌ DISCARDED (below {quality_threshold})')

        except Exception as e:
            print(f'  ❌ Error: {e}')
            continue

    print(f'\n{"="*50}')
    print(f'Generated {len(synthetic_pairs)} / {len(monolingual_sentences)} quality pairs')
    print(f'Acceptance rate: {len(synthetic_pairs)/max(len(monolingual_sentences),1)*100:.1f}%')
    if synthetic_pairs:
        avg_rt = sum(p["rt_score"] for p in synthetic_pairs) / len(synthetic_pairs)
        print(f'Mean round-trip score: {avg_rt:.3f}')
    return synthetic_pairs


# ── Save synthetic dataset for fine-tuning ────────────────────────────────────
def save_synthetic_dataset(pairs: list[dict], lang_name: str) -> str:
    """
    Save synthetic pairs in instruction fine-tuning JSONL format.
    Ready to feed directly into SFTTrainer.
    """
    output_path = DATA_DIR / f'synthetic_{lang_name.lower()}_hi.jsonl'
    with open(output_path, 'w', encoding='utf-8') as f:
        for pair in pairs:
            record = {
                'instruction': pair['instruction'],
                'input':       pair['input'],
                'output':      pair['output'],
                'metadata': {
                    'lang_pair': pair['lang_pair'],
                    'rt_score':  pair['rt_score'],
                    'source':    'back_translation',
                }
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    print(f'\n✅ Saved {len(pairs)} pairs to {output_path}')
    return str(output_path)


print('✅ Back-translation pipeline defined')
print('   Supports: Bhojpuri (bho) · Maithili (mai) · Awadhi (awa)')

✅ Back-translation pipeline defined
   Supports: Bhojpuri (bho) · Maithili (mai) · Awadhi (awa)


In [ ]:
# Run back-translation for Bhojpuri
# (uses ~5 Groq API calls — runs quickly on free tier)

bhojpuri_config = LOW_RESOURCE_CONFIG['bho']

synthetic_pairs = back_translation_pipeline(
    monolingual_sentences=bhojpuri_config['sample_sentences'],
    source_lang_name=bhojpuri_config['name'],
    quality_threshold=0.45,   # lower threshold for low-resource
    max_pairs=5,
)

if synthetic_pairs:
    dataset_path = save_synthetic_dataset(synthetic_pairs, 'Bhojpuri')
    print(f'\nSample synthetic pair:')
    print(f"  Bhojpuri: {synthetic_pairs[0]['source']}")
    print(f"  Hindi   : {synthetic_pairs[0]['pivot_translation']}")
    print(f"  RT score: {synthetic_pairs[0]['rt_score']:.3f}")


Back-translation pipeline for Bhojpuri
Input sentences: 5
Quality threshold: 0.45

Processing [1/5]: हम बाजार जाइत बानी।
  → Hindi pivot: हम बाजार जा रहे हैं।
  → Back-translated: हम बाजार जा रहल बानी।
  → Round-trip similarity: 0.960 ✅ KEPT

Processing [2/5]: ऊ लोग खाना खात बाड़न।
  → Hindi pivot: वे लोग खाना खा रहे हैं।
  → Back-translated: उहे लोग खाना खाता बा।
  → Round-trip similarity: 0.911 ✅ KEPT

Processing [3/5]: रउरा के नाम का बा?
  → Hindi pivot: तुम्हारा नाम क्या है?
  → Back-translated: तुम्हार नाम का ह ?
  → Round-trip similarity: 0.705 ✅ KEPT

Processing [4/5]: आज मौसम बहुत नीमन बा।
  → Hindi pivot: आज मौसम बहुत सुहावना है।
  → Back-translated: आज मौसम बहुत सुहावना बा
  → Round-trip similarity: 0.886 ✅ KEPT

Processing [5/5]: हमार घर इहाँ बा।
  → Hindi pivot: हमारा घर यहाँ है।
  → Back-translated: हमार घर इहाँ ह।
  → Round-trip similarity: 0.989 ✅ KEPT

Generated 5 / 5 quality pairs
Acceptance rate: 100.0%
Mean round-trip score: 0.890

✅ Saved 5 pairs to /content/transl

## Step 14 — Fine-Tuning Pipeline (QLoRA + SFTTrainer)

In [ ]:
PROMPT_TEMPLATE = (
    'Below is an instruction to translate text into Hindi.\n\n'
    '### Instruction:\n{instruction}\n\n'
    '### Input:\n{input}\n\n'
    '### Response:\n{output}'
)

INFERENCE_TEMPLATE = (
    'Below is an instruction to translate text into Hindi.\n\n'
    '### Instruction:\nTranslate to Hindi:\n\n'
    '### Input:\n{input}\n\n'
    '### Response:\n'
)


def load_combined_dataset(jsonl_paths: list[str]):
    """Load and merge multiple JSONL datasets (Samanantar, IIT Bombay, synthetic)."""
    from datasets import Dataset
    all_records = []
    for path in jsonl_paths:
        if not Path(path).exists():
            print(f'Warning: {path} not found, skipping')
            continue
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    record = json.loads(line)
                    all_records.append({'text': PROMPT_TEMPLATE.format(
                        instruction=record.get('instruction', 'Translate to Hindi:'),
                        input=record['input'],
                        output=record['output'],
                    )})
    import random; random.shuffle(all_records)
    print(f'Combined dataset: {len(all_records)} examples')
    return Dataset.from_list(all_records)


def fine_tune_model(
    base_model: str,
    data_paths: list[str],
    output_dir: str,
    epochs: int = 3,
    batch_size: int = 4,
    max_seq_length: int = 512,
    learning_rate: float = 2e-4,
) -> None:
    """
    QLoRA fine-tuning on combined En→Hi + synthetic low-resource data.

    Recommended base models:
      'google/gemma-2b'           — 4GB VRAM, fast
      'mistralai/Mistral-7B-v0.3' — 8GB VRAM, better quality
      'AI4Bharat/indic-llm-13b'   — 12GB VRAM, best for Indic
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig, TaskType
    from trl import SFTTrainer, SFTConfig

    print(f'Loading base model: {base_model}')

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    model = AutoModelForCausalLM.from_pretrained(
        base_model, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True, torch_dtype=torch.float16,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    dataset = load_combined_dataset(data_paths)

    training_args = SFTConfig(
        output_dir=output_dir, num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=max(1, 16 // batch_size),
        optim='paged_adamw_32bit', learning_rate=learning_rate,
        weight_decay=0.001, fp16=True, max_grad_norm=0.3,
        warmup_ratio=0.03, lr_scheduler_type='cosine',
        logging_steps=10, save_strategy='epoch',
        group_by_length=True, max_seq_length=max_seq_length,
        dataset_text_field='text', report_to='none',
    )

    trainer = SFTTrainer(
        model=model, train_dataset=dataset,
        args=training_args, tokenizer=tokenizer,
    )

    print('Starting fine-tuning...')
    trainer.train()

    final_path = f'{output_dir}/final_model'
    trainer.model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f'✅ Fine-tuning complete. Saved to {final_path}')


# Create sample training data (replace with real datasets)
sample_data = [
    {'instruction': 'Translate to Hindi:', 'input': 'Good morning!', 'output': 'शुभ प्रभात!'},
    {'instruction': 'Translate to Hindi:', 'input': 'How are you?', 'output': 'आप कैसे हैं?'},
    {'instruction': 'Translate to Hindi:', 'input': 'The Supreme Court ruled today.', 'output': 'सर्वोच्च न्यायालय ने आज फ़ैसला सुनाया।'},
    {'instruction': 'Translate to Hindi:', 'input': 'Please submit the report by tomorrow.', 'output': 'कृपया कल तक रिपोर्ट जमा करें।'},
]

SAMPLE_DATA_PATH = DATA_DIR / 'sample_en_hi.jsonl'
with open(SAMPLE_DATA_PATH, 'w', encoding='utf-8') as f:
    for r in sample_data:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Add synthetic back-translation data if available
data_paths = [str(SAMPLE_DATA_PATH)]
synthetic_path = DATA_DIR / 'synthetic_bhojpuri_hi.jsonl'
if synthetic_path.exists():
    data_paths.append(str(synthetic_path))

print('✅ Fine-tuning pipeline ready')
print(f'   Training files: {data_paths}')
print()
print('# To run fine-tuning, uncomment:')
print('# fine_tune_model(')
print('#     base_model  = "google/gemma-2b",')
print('#     data_paths  = data_paths,')
print('#     output_dir  = "/content/hindi_lora_v2",')
print('#     epochs      = 3,')
print('#     batch_size  = 4,')
print('# )')

✅ Fine-tuning pipeline ready
   Training files: ['/content/translation_data_v2/sample_en_hi.jsonl', '/content/translation_data_v2/synthetic_bhojpuri_hi.jsonl']

# To run fine-tuning, uncomment:
# fine_tune_model(
#     base_model  = "google/gemma-2b",
#     data_paths  = data_paths,
#     output_dir  = "/content/hindi_lora_v2",
#     epochs      = 3,
#     batch_size  = 4,
# )


## Step 15 — Benchmark Evaluation

In [ ]:
BENCHMARK = [
    {
        'source': 'Narendra Modi addressed Parliament of India yesterday morning.',
        'reference': 'नरेंद्र मोदी ने कल सुबह भारतीय संसद को संबोधित किया।',
        'category': 'entity+temporal',
    },
    {
        'source': 'The Reserve Bank of India will review interest rates next month.',
        'reference': 'भारतीय रिज़र्व बैंक अगले महीने ब्याज दरों की समीक्षा करेगा।',
        'category': 'entity+temporal',
    },
    {
        'source': 'Sachin Tendulkar broke the ice at the press conference last evening.',
        'reference': 'सचिन तेंदुलकर ने कल शाम प्रेस कॉन्फ्रेंस में बात की शुरुआत की।',
        'category': 'idiom+entity',
    },
    {
        'source': 'Please submit your application to the Income Tax Department by tomorrow.',
        'reference': 'कृपया कल तक आयकर विभाग को अपना आवेदन जमा करें।',
        'category': 'formal+temporal',
    },
    {
        'source': 'ISRO launched a satellite from the Bengaluru facility this morning.',
        'reference': 'इसरो ने आज सुबह बेंगलुरु सुविधा से एक उपग्रह लॉन्च किया।',
        'category': 'entity+technical',
    },
]

print('Running V2 benchmark evaluation...\n')
print(f'{"Source":<45} {"Cat":<15} {"BERT":>6} {"Agg":>6} {"Iters":>5}')
print('-' * 85)

all_results = []
for item in BENCHMARK:
    r = translate(
        source_text=item['source'],
        reference_translations=[item['reference']],
        max_iterations=3, pass_threshold=0.58,
    )
    all_results.append((item, r))
    s = r['final_scores']
    print(
        f"{item['source'][:43]:<45} {item['category']:<15} "
        f"{s.get('bert_score_f1',0):6.3f} "
        f"{s.get('aggregate',0):6.3f} "
        f"{r['iterations']:5d}"
    )

avg_agg  = sum(r['final_scores'].get('aggregate',0) for _,r in all_results) / len(all_results)
avg_bert = sum(r['final_scores'].get('bert_score_f1',0) for _,r in all_results) / len(all_results)
avg_iter = sum(r['iterations'] for _,r in all_results) / len(all_results)

print('-' * 85)
print(f'{'AVERAGES':<45} {'':15} {avg_bert:6.3f} {avg_agg:6.3f} {avg_iter:5.1f}')
print(f'\n✅ V2 Benchmark complete')
print(f'   Mean aggregate score : {avg_agg:.3f}')
print(f'   Mean BERTScore       : {avg_bert:.3f}')
print(f'   Mean iterations      : {avg_iter:.1f}')

Running V2 benchmark evaluation...

Source                                        Cat               BERT    Agg Iters
-------------------------------------------------------------------------------------

[Node 1 — Input] 'Narendra Modi addressed Parliament of India yesterday morning.'
[Node 2 — Enrichment] entities=3 temporal=2 senses=5
[Node 3 — KG] facts=2 kg_context_len=139
[Node 4 — Embedding] dim=768 epi=1 sem=5
[Node 5 — Translation] iter=1: 'नरेंद्र मोदी ने कल सुबह भारतीय संसद को संबोधित किया।'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 6 — Evaluator] ✅ PASSED agg=0.999 BLEU=100.0 BERT=1.000
[Node 7 — Feedback] Translation accepted. aggregate=0.999 BLEU=100.0 BERTScore=1.000
[Node 8 — Memory] episodic=4 semantic=29 kg_updated=True
[Node 9 — Output] 'नरेंद्र मोदी ने कल सुबह भारतीय संसद को संबोधित किया।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │ 100.00 │ 100.00 │   1.0000  │ 0.9946  │  0.9987

Agent trace: input_node → enrichment_node → kg_node → embedding_node → translation_node_iter1 → evaluator_node → feedback_node → memory_update_node
Narendra Modi addressed Parliament of India   entity+temporal  1.000  0.999     1

[Node 1 — Input] 'The Reserve Bank of India will review interest rates next month.'
[Node 2 — Enrichment] entities=2 temporal=1 senses=7
[Node 3 — KG] facts=0 kg_context_len=71
[Node 4 — Embedding] dim=768 epi=1 sem=5
[Node 5 — Translation] iter=1: 'रिज़र्व बैंक ऑफ़ इंडिया अगले महीने ब्याज दरों की समीक्षा करे...'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 6 — Evaluator] ✅ PASSED agg=0.750 BLEU=26.5 BERT=0.924
[Node 7 — Feedback] Translation accepted. aggregate=0.750 BLEU=26.54 BERTScore=0.924
[Node 8 — Memory] episodic=7 semantic=32 kg_updated=True
[Node 9 — Output] 'कृपया अपना आवेदन आयकर विभाग को कल तक जमा करें।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  26.54 │  70.48 │   0.9242  │ 0.9757  │  0.7505

Agent trace: input_node → enrichment_node → kg_node → embedding_node → translation_node_iter1 → evaluator_node → feedback_node → memory_update_node
Please submit your application to the Incom   formal+temporal  0.924  0.750     1

[Node 1 — Input] 'ISRO launched a satellite from the Bengaluru facility this morning.'
[Node 2 — Enrichment] entities=3 temporal=1 senses=4
[Node 3 — KG] facts=2 kg_context_len=79
[Node 4 — Embedding] dim=768 epi=0 sem=5
[Node 5 — Translation] iter=1: 'इसरो ने बेंगलुरु सुविधा से इस सुबह एक उपग्रह प्रक्षेपित किया...'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Node 6 — Evaluator] ✅ PASSED agg=0.714 BLEU=21.8 BERT=0.904
[Node 7 — Feedback] Translation accepted. aggregate=0.714 BLEU=21.83 BERTScore=0.904
[Node 8 — Memory] episodic=8 semantic=33 kg_updated=True
[Node 9 — Output] 'इसरो ने बेंगलुरु सुविधा से इस सुबह एक उपग्रह प्रक्षेपित किया।'

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │  21.83 │  59.65 │   0.9039  │ 1.0000  │  0.7140

Agent trace: input_node → enrichment_node → kg_node → embedding_node → translation_node_iter1 → evaluator_node → feedback_node → memory_update_node
ISRO launched a satellite from the Bengalur   entity+technical  0.904  0.714     1
-------------------------------------------------------------------------------------
AVERAGES                                                       0.943  0.824   1.0

✅ V2 Benchmark complete
   Mean aggregate score : 0.824
   Mean BERTScore       : 0.943
   Mean iterations      : 1.0


In [ ]:
# Detailed per-sentence analysis
for item, result in all_results:
    print(f"\n{'='*65}")
    print(f"Category : {item['category']}")
    print(f"English  : {item['source']}")
    print(f"Reference: {item['reference']}")
    print(f"Output   : {result['final_translation']}")
    print(f"Entities : {[(e['text'], e['type'], e.get('hindi','?')) for e in result['named_entities']]}")
    print(f"Temporal : {[(t['text'], t.get('hindi','?')) for t in result['temporal_expressions']]}")
    print(f"KG facts : {len(result['kg_facts'])}")
    print(f"\n{format_score_table(result['score_history'])}")


Category : entity+temporal
English  : Narendra Modi addressed Parliament of India yesterday morning.
Reference: नरेंद्र मोदी ने कल सुबह भारतीय संसद को संबोधित किया।
Output   : नरेंद्र मोदी ने कल सुबह भारतीय संसद को संबोधित किया।
Entities : [('Narendra Modi', 'PERSON', 'नरेंद्र मोदी'), ('Parliament of India', 'ORG', 'भारतीय संसद'), ('yesterday morning', 'TIME', '')]
Temporal : [('yesterday', 'कल'), ('morning', 'सुबह')]
KG facts : 2

Iter │  BLEU  │  chrF  │ BERTScore │  COMET  │ Aggregate
─────┼────────┼────────┼───────────┼─────────┼──────────
   1 │ 100.00 │ 100.00 │   1.0000  │ 0.9946  │  0.9987

Category : entity+temporal
English  : The Reserve Bank of India will review interest rates next month.
Reference: भारतीय रिज़र्व बैंक अगले महीने ब्याज दरों की समीक्षा करेगा।
Output   : रिज़र्व बैंक ऑफ़ इंडिया अगले महीने ब्याज दरों की समीक्षा करेगा।
Entities : [('The Reserve Bank of India', 'ORG', ''), ('next month', 'DATE', '')]
Temporal : [('next month', 'अगले महीने')]
KG facts : 0

Iter │

## Step 16 — Memory Inspection & KG Visualisation

In [ ]:
# Knowledge Graph statistics
print('='*50)
print('KNOWLEDGE GRAPH STATS')
print('='*50)
G = knowledge_graph.G
print(f'Total nodes  : {G.number_of_nodes()}')
print(f'Total edges  : {G.number_of_edges()}')

# Count by node type
node_types = {}
for _, data in G.nodes(data=True):
    nt = data.get('node_type', data.get('type', 'ENTITY'))
    node_types[nt] = node_types.get(nt, 0) + 1
print(f'Node types   : {node_types}')

# Count by edge type
edge_types = {}
for _, _, data in G.edges(data=True):
    et = data.get('relation', 'unknown')
    edge_types[et] = edge_types.get(et, 0) + 1
print(f'Edge types   : {edge_types}')

print('\n' + '='*50)
print('MEMORY STATS')
print('='*50)
wm = working_memory.to_dict()
print(f'Working memory entries  : {len(wm["history"])}')
print(f'Entity cache entries    : {len(wm["entity_cache"])}')
print(f'Episodic episodes       : {episodic_memory.count()}')
print(f'Semantic chunks         : {semantic_memory.count()}')

# MCP tools registered
print('\n' + '='*50)
print('MCP TOOL REGISTRY')
print('='*50)
for name in MCP_TOOL_REGISTRY:
    print(f'  • {name}')

# Sample entity lookups
print('\n' + '='*50)
print('SAMPLE KG LOOKUPS')
print('='*50)
test_entities = ['Supreme Court', 'Narendra Modi', 'Mumbai', 'ISRO']
for ent in test_entities:
    hindi = knowledge_graph.get_hindi_translation(ent)
    print(f'  {ent:25s} → {hindi or "(not in KG)"}')

KNOWLEDGE GRAPH STATS
Total nodes  : 71
Total edges  : 30
Node types   : {'PERSON': 8, 'ORG': 14, 'GPE': 4, 'LOC': 14, 'TIMEX': 10, 'GRAMMAR_RULE': 5, 'SYNSET': 16}
Edge types   : {'translates_to': 20, 'is_a': 4, 'has_type': 6}

MEMORY STATS
Working memory entries  : 8
Entity cache entries    : 6
Episodic episodes       : 8
Semantic chunks         : 33

MCP TOOL REGISTRY
  • extract_entities
  • extract_temporal_expressions
  • query_knowledge_graph
  • translate_to_hindi
  • evaluate_translation
  • generate_feedback
  • update_memory

SAMPLE KG LOOKUPS
  Supreme Court             → सर्वोच्च न्यायालय
  Narendra Modi             → नरेंद्र मोदी
  Mumbai                    → मुंबई
  ISRO                      → इसरो


## Step 17 — Export Results

In [ ]:
from datetime import datetime

REPORT = {
    'timestamp':     datetime.now().isoformat(),
    'session_id':    SESSION_ID,
    'model':         os.getenv('TRANSLATION_MODEL'),
    'version':       'V2',
    'directions':    ['NLP Enrichment', 'Knowledge Graph', 'Multi-Agent MCP', 'Low-Resource Back-Translation'],
    'memory_stats': {
        'working_entries':   len(working_memory.to_dict()['history']),
        'episodic_episodes': episodic_memory.count(),
        'semantic_chunks':   semantic_memory.count(),
        'kg_nodes':          knowledge_graph.G.number_of_nodes(),
        'kg_edges':          knowledge_graph.G.number_of_edges(),
    },
    'mcp_tools': list(MCP_TOOL_REGISTRY.keys()),
    'benchmark': [
        {
            'source':       item['source'],
            'category':     item['category'],
            'reference':    item['reference'],
            'translation':  result['final_translation'],
            'final_scores': result['final_scores'],
            'iterations':   result['iterations'],
            'entities':     result['named_entities'],
            'temporal':     result['temporal_expressions'],
            'kg_facts_count': len(result['kg_facts']),
            'agent_trace':  result['agent_trace'],
        }
        for item, result in all_results
    ],
    'summary': {
        'mean_aggregate':  round(avg_agg, 3),
        'mean_bertscore':  round(avg_bert, 3),
        'mean_iterations': round(avg_iter, 1),
    }
}

REPORT_PATH = DATA_DIR / 'evaluation_report_v2.json'
REPORT_PATH.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2))
print(f'✅ Report saved to {REPORT_PATH}')

try:
    from google.colab import files
    files.download(str(REPORT_PATH))
    print('✅ Downloaded to local machine')
except ImportError:
    print(f'(Not in Colab — file at: {REPORT_PATH})')

print('\n' + '='*60)
print('V2 SESSION SUMMARY')
print('='*60)
print(f'Directions implemented : 4 (NLP·KG·MultiAgent·LowResource)')
print(f'MCP tools registered   : {len(MCP_TOOL_REGISTRY)}')
print(f'KG nodes               : {knowledge_graph.G.number_of_nodes()}')
print(f'Episodic episodes      : {episodic_memory.count()}')
print(f'Semantic chunks        : {semantic_memory.count()}')
print(f'Mean aggregate score   : {avg_agg:.3f}')
print(f'Mean iterations        : {avg_iter:.1f}')

✅ Report saved to /content/translation_data_v2/evaluation_report_v2.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded to local machine

V2 SESSION SUMMARY
Directions implemented : 4 (NLP·KG·MultiAgent·LowResource)
MCP tools registered   : 7
KG nodes               : 71
Episodic episodes      : 8
Semantic chunks        : 33
Mean aggregate score   : 0.824
Mean iterations        : 1.0
